# 00 — Загрузка данных через Kaggle API

Этот ноутбук демонстрирует **самостоятельное получение данных** через [Kaggle Public API](https://www.kaggle.com/docs/api).

Вместо ручного скачивания файла с сайта мы:
1. Получаем метаданные датасета через REST API
2. Скачиваем файл программно (два метода: через `kaggle` lib или через `requests`)
3. Проверяем целостность скачанного файла

**Датасет:** [Student Mental Health and Burnout Dataset](https://www.kaggle.com/datasets/sehaj1104/student-mental-health-and-burnout-dataset)  
**owner_slug:** `sehaj1104`  
**dataset_slug:** `student-mental-health-and-burnout-dataset`

## 0. Настройка аутентификации

Kaggle API требует токен. Получить его: **kaggle.com → Account → Settings → API → Create New Token**.

После получения токена создайте файл .env в корне проекта.

.env:
```
   KAGGLE_USERNAME=your_username
   KAGGLE_KEY=your_api_key
```

In [1]:
import sys
sys.path.append('..')

import os
import pandas as pd
from pathlib import Path

# Создай файл .env:
#   KAGGLE_USERNAME=your_username
#   KAGGLE_KEY=your_api_key

from src.data_loader import KaggleDataLoader
print('KaggleDataLoader импортирован успешно')

KaggleDataLoader импортирован успешно


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Получение метаданных датасета через REST API

Перед скачиванием получим информацию о датасете: размер, лицензия, теги, количество скачиваний.

In [2]:
loader = KaggleDataLoader(
    owner="sehaj1104",
    dataset="student-mental-health-and-burnout-dataset",
    output_dir="../data/raw"
)

loader.print_dataset_info()   # метаданные и список файлов

KAGGLE DATASET INFO


REST metadata failed: 404 Client Error: Not Found for url: https://www.kaggle.com/api/v1/datasets/sehaj1104/student-mental-health-and-burnout-dataset/metadata


Title         : sehaj1104/student-mental-health-and-burnout-dataset
URL           : https://www.kaggle.com/datasets/sehaj1104/student-mental-health-and-burnout-dataset
Note          : Полные метаданные временно недоступны. Откройте URL для подробностей.
------------------------------------------------------------
Files in dataset:
  student_mental_health_burnout.csv           12245.9 KB


In [3]:
# Получить метаданные как словарь для дальнейшего использования
meta = loader.fetch_metadata()
print('Метаданные датасета:')
for k, v in meta.items():
    print(f'  {k:<20}: {v}')

REST metadata failed: 404 Client Error: Not Found for url: https://www.kaggle.com/api/v1/datasets/sehaj1104/student-mental-health-and-burnout-dataset/metadata


Метаданные датасета:
  title               : sehaj1104/student-mental-health-and-burnout-dataset
  url                 : https://www.kaggle.com/datasets/sehaj1104/student-mental-health-and-burnout-dataset
  note                : Полные метаданные временно недоступны. Откройте URL для подробностей.


## 2. Список файлов в датасете

После скачивания датасета класс анализирует локальный кэш `kagglehub` и показывает список файлов, содержащихся в датасете.


In [4]:
files = loader.list_dataset_files()
print(f'Файлов в датасете: {len(files)}')
for f in files:
    size_mb = f['size_bytes'] / (1024**2)
    print(f"  {f['name']:<50} {size_mb:.2f} MB")

Файлов в датасете: 1
  student_mental_health_burnout.csv                  11.96 MB


## 3. Скачивание датасета

Скачивание выполняется через библиотеку `kagglehub`.

Во время загрузки:
- Kaggle автоматически кэширует датасет;
- проект ищет CSV‑файл внутри скачанного набора;
- найденный CSV копируется в директорию `data/raw`.

Если датасет уже был скачан ранее, повторная загрузка не выполняется.

Если загрузка не удалась, делаем прямой REST API запрос через `requests`.

In [5]:
# Скачать датасет (auto: kaggle lib -> requests fallback)
csv_path = loader.download()
print(f'\nПуть к файлу: {csv_path}')
print(f'Размер файла: {csv_path.stat().st_size / (1024**2):.1f} MB')

Downloaded to: C:\Users\User\.cache\kagglehub\datasets\sehaj1104\student-mental-health-and-burnout-dataset\versions\1
✓ Saved to ..\data\raw\student_mental_health_burnout.csv

Путь к файлу: ..\data\raw\student_mental_health_burnout.csv
Размер файла: 12.0 MB


## 4. Проверка скачанного файла

In [6]:
# Проверяем, что файл скачался корректно
df = pd.read_csv('../data/raw/student_mental_health_burnout.csv')

print('=== Базовая информация о датасете ===')
print(f'Строк:   {df.shape[0]:,}')
print(f'Колонок: {df.shape[1]}')
print(f'\nКолонки: {df.columns.tolist()}')
print(f'\nТипы данных:\n{df.dtypes}')
print(f'\nПервые 3 строки:')
df.head(3)

=== Базовая информация о датасете ===
Строк:   150,000
Колонок: 20

Колонки: ['student_id', 'age', 'gender', 'course', 'year', 'daily_study_hours', 'daily_sleep_hours', 'screen_time_hours', 'stress_level', 'anxiety_score', 'depression_score', 'academic_pressure_score', 'financial_stress_score', 'social_support_score', 'physical_activity_hours', 'sleep_quality', 'attendance_percentage', 'cgpa', 'internet_quality', 'burnout_level']

Типы данных:
student_id                   int64
age                          int64
gender                      object
course                      object
year                        object
daily_study_hours          float64
daily_sleep_hours          float64
screen_time_hours          float64
stress_level                object
anxiety_score                int64
depression_score             int64
academic_pressure_score      int64
financial_stress_score       int64
social_support_score         int64
physical_activity_hours    float64
sleep_quality              

,student_id,age,gender,course,year,daily_study_hours,daily_sleep_hours,screen_time_hours,stress_level,anxiety_score,depression_score,academic_pressure_score,financial_stress_score,social_support_score,physical_activity_hours,sleep_quality,attendance_percentage,cgpa,internet_quality,burnout_level
0,100001,23,Male,BTech,1st,4.3,6.8,6.1,High,10,3,4,2,6,1.8,Average,66.5,9.63,Good,High
1,100002,20,Male,BTech,3rd,1.4,4.7,3.0,High,2,10,8,5,9,1.9,Poor,55.8,6.04,Poor,Low
2,100003,24,Female,BCA,4th,3.7,4.8,1.5,Low,2,7,8,6,3,0.8,Good,85.0,8.31,Good,High


In [7]:
# Проверка целостности: нет ли пустого файла или битых данных
assert df.shape[0] > 10_000, 'Датасет слишком мал — возможно, скачался частично'
assert df.shape[1] >= 10,   'Слишком мало колонок'
assert 'burnout_level' in df.columns or any('burnout' in c.lower() for c in df.columns), \
    'Целевая переменная не найдена'

print('✓ Все проверки пройдены — датасет загружен корректно!')
print(f'  Размер: {df.shape[0]:,} строк × {df.shape[1]} колонок')
print(f'  Память: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB')

✓ Все проверки пройдены — датасет загружен корректно!
  Размер: 150,000 строк × 20 колонок
  Память: 68.2 MB


## Итог

Данные успешно загружены через **Kaggle Public API** программным способом.

Использованные эндпоинты:
- `GET /api/v1/datasets/{owner}/{dataset}` — метаданные датасета
- `GET /api/v1/datasets/{owner}/{dataset}/versions/1/files` — список файлов
- `GET /api/v1/datasets/download/{owner}/{dataset}` — скачивание ZIP-архива

Далее: **[01_eda.ipynb](01_eda.ipynb)** — EDA и предобработка данных.